# P16 — Sistemas agentic contemporáneos: memoria, reflexión, multiagente e interoperabilidad

## 1. Título y paper

**Paper:** *Sistemas agentic contemporáneos (nodo de frontera, revisable)*  
**Autoría:** Varios (nodo compuesto)  
**Año y venue:** 2023 · Conjunto de trabajos posteriores a ReAct — releer con fecha de consulta  
**Nivel:** L5 · **Motor:** `agentic`  
**Ficha completa:** [`P16_agentic_systems`](../../papers/foundational/P16_agentic_systems/README.md)

**Hito:** El agente deja de ser un bucle y pasa a ser un sistema: memoria, reflexión, planificación, presupuesto, múltiples agentes y protocolos de interoperabilidad.

- [Shinn et al. (2023), Reflexion](https://arxiv.org/abs/2303.11366)
- [Park et al. (2023), Generative Agents](https://arxiv.org/abs/2304.03442)
- [Wang et al. (2023), Voyager](https://arxiv.org/abs/2305.16291)
- [Wu et al. (2023), AutoGen](https://arxiv.org/abs/2308.08155)
- [Model Context Protocol (especificación)](https://modelcontextprotocol.io)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Un bucle ReAct sin memoria, sin criterio de parada ni presupuesto no sobrevive a tareas largas ni a fallos de herramienta.
2. Ejecutar una implementación mínima de la propuesta: No hay una única propuesta: hay una familia de trabajos que añaden autocrítica, memoria episódica, currículo autónomo, orquestación multiagente y estándares de acceso a herramientas.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P13
- P14
- P15


## 4. Intuición

Un agente que funciona en una demo y falla en producción casi nunca falla por el modelo: falla porque no tenía presupuesto, ni criterio de parada, ni memoria, ni un plan para el momento en que una herramienta devuelve un error.


## 5. Concepto mínimo

Un sistema agentic contemporáneo se describe por sus componentes, no por su prompt:

```text
plan · herramientas tipadas · memoria · presupuesto · criterio de parada · escalamiento
```

Los trabajos posteriores a ReAct añaden autocrítica (Reflexion), memoria episódica con recuperación (Generative Agents), currículo autónomo (Voyager), orquestación multiagente (AutoGen) y estandarización del acceso a herramientas (MCP).


## 6. Código explicado

El motor ejecuta un agente con presupuesto explícito que se topa con un fallo de herramienta.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('agentic', seed=7)['result']
show(r['presupuesto'])
show(r['consumido'])
print('\ntraza:')
for paso in r['traza']:
    print(' ', paso)
print('\nescalado a humano:', r['escalado_a_humano'])

## 7. Predicción antes de ejecutar

1. ¿Agotará el agente su presupuesto de pasos o se detendrá antes?
2. Cuando la verificación falle, ¿debe reintentar, seguir sin verificar, o parar?
3. ¿Qué componente del sistema es el que hace auditable esta ejecución?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
componentes = r['componentes']
riesgo_si_falta = {
    'plan': 'el agente deambula sin objetivo verificable',
    'herramientas': 'el modelo alucina la acción en lugar de ejecutarla',
    'memoria': 'repite trabajo y pierde el contexto entre pasos',
    'presupuesto': 'coste ilimitado ante un bucle',
    'criterio de parada': 'nunca termina; consume hasta el timeout',
    'escalamiento': 'un fallo se convierte en una respuesta inventada',
}
for c in componentes:
    print(f'{c:<20} → si falta: {riesgo_si_falta[c]}')

## 9. Salida interpretable

El agente se detiene en el paso de verificación y escala en lugar de responder igualmente. **Parar es un resultado correcto.** Un sistema que siempre devuelve una respuesta está ocultando sus fallos, no evitándolos.


## 10. Comentario pedagógico

Este nodo es el más volátil del eje y por eso vive con fecha de consulta. Lo estable son las preguntas (¿quién define el objetivo? ¿quién paga el presupuesto? ¿quién responde por el error?); lo inestable son los nombres de framework de cada temporada.


## 11. Error o anti-patrón deliberado

Anti-patrón: «agente autónomo» sin límite de gasto ni permisos, evaluado por si «funcionó una vez».


In [ ]:
demo = {'ejecuciones': 1, 'exito': True, 'conclusion': 'listo para producción'}
show(demo)
print('→ n=1 no es evidencia. No hay varianza, ni casos límite, ni fallos de herramienta,')
print('  ni coste medido, ni comportamiento ante entradas adversarias.')

## 12. Corrección

Un reporte mínimamente serio de un agente incluye distribución, no una anécdota:


In [ ]:
reporte = {
    'ejecuciones': 100,
    'tasa_de_exito': 0.71,
    'tasa_de_escalamiento_correcto': 0.18,
    'tasa_de_respuesta_inventada': 0.03,
    'coste_medio_por_tarea': '0.9 llamadas de herramienta · 4 pasos',
    'p95_pasos': 9,
    'casos_adversarios_probados': ['herramienta caída', 'salida malformada', 'instrucción inyectada'],
}
show(reporte)

## 13. Desafío guiado

Reduce el presupuesto a 2 pasos y comprueba que el agente aborta de forma limpia en lugar de fallar a medias.


In [ ]:
presupuesto = {'pasos': 2}
plan = ['leer_requisito', 'consultar_datos', 'verificar', 'responder']
ejecutados = []
for paso in plan:
    if len(ejecutados) >= presupuesto['pasos']:
        print(f'ABORTADO antes de «{paso}» · pasos ejecutados: {ejecutados}')
        break
    ejecutados.append(paso)
else:
    print('completado:', ejecutados)

## 14. Desafío autónomo

Toma un agente que hayas construido en la parte 09 del programa y añádele los seis componentes. Ejecútalo 50 veces sobre 10 tareas y reporta tasa de éxito, de escalamiento y de respuesta inventada, con al menos tres casos adversarios. Fecha el informe.


## 15. Evidencia de aprendizaje

Guarda la traza con presupuesto, la tabla componente→riesgo-si-falta y el reporte de evaluación con distribución en lugar de anécdota.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P16_agentic_systems/README.md) · evaluación formal: [`assessments/papers/P16_agentic_systems.md`](../../assessments/papers/P16_agentic_systems.md)


## 16. Cierre

Aquí termina la ruta mínima y empieza la frontera. Lo que sigue no está consolidado: se registra en `frontier/current-topics.yaml` con fecha y fuente, y se relee, no se cita como firme.


## 17. Conexión con el siguiente hito

- frontier/current-topics.yaml

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
